# Parameter Golf — Download Data

Downloads FineWeb training + validation data to Google Drive so it persists across sessions.
Run once — all other notebooks use the cached data from Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA = '/content/drive/MyDrive/parameter-golf/data'
os.makedirs(DRIVE_DATA, exist_ok=True)

In [ ]:
# Clone official repo for the download script
%cd /content
if not os.path.exists('/content/parameter-golf'):
    !git clone --depth 1 https://github.com/openai/parameter-golf.git
!pip install -q sentencepiece huggingface-hub datasets tqdm

In [ ]:
# How many training shards to download?
# 1 shard  = ~100M tokens (~200MB) — smoke tests only
# 10 shards = ~1B tokens (~2GB) — fast experiments (model cycles data 5x)
# 80 shards = ~8B tokens (~16GB) — FULL dataset (matches competition, no cycling)
#
# For valid error analysis, use 80 (the model's real limits depend on seeing
# the full data distribution, not a subset it memorizes through repetition)

TRAIN_SHARDS = 80

In [ ]:
# Download to local disk first (fast), then copy to Drive (persistent)
%cd /content/parameter-golf
!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards {TRAIN_SHARDS}

# Check what was downloaded
import glob
train_files = sorted(glob.glob('data/datasets/fineweb10B_sp1024/fineweb_train_*.bin'))
val_files = sorted(glob.glob('data/datasets/fineweb10B_sp1024/fineweb_val_*.bin'))
tok_files = sorted(glob.glob('data/tokenizers/*'))
print(f'Train shards: {len(train_files)}')
print(f'Val shards: {len(val_files)}')
print(f'Tokenizer files: {tok_files}')

In [ ]:
# Copy to Drive for persistence
import shutil

src_data = '/content/parameter-golf/data/datasets/fineweb10B_sp1024'
src_tok = '/content/parameter-golf/data/tokenizers'
dst_data = f'{DRIVE_DATA}/datasets/fineweb10B_sp1024'
dst_tok = f'{DRIVE_DATA}/tokenizers'

os.makedirs(dst_data, exist_ok=True)
os.makedirs(dst_tok, exist_ok=True)

# Copy data files (skip existing)
copied = 0
for f in glob.glob(f'{src_data}/*.bin'):
    dst = f'{dst_data}/{os.path.basename(f)}'
    if not os.path.exists(dst):
        print(f'Copying {os.path.basename(f)}...')
        shutil.copy2(f, dst)
        copied += 1

# Copy tokenizer
for f in glob.glob(f'{src_tok}/*'):
    dst = f'{dst_tok}/{os.path.basename(f)}'
    if not os.path.exists(dst):
        shutil.copy2(f, dst)
        copied += 1

print(f'\nCopied {copied} new files to Drive')
print(f'Drive data: {DRIVE_DATA}')

# Verify
drive_train = glob.glob(f'{dst_data}/fineweb_train_*.bin')
drive_val = glob.glob(f'{dst_data}/fineweb_val_*.bin')
print(f'Drive train shards: {len(drive_train)}')
print(f'Drive val shards: {len(drive_val)}')